# Xuất YOLOv5n sang TFLite · Signa v1.1

Mở trên Colab trong runtime **CPU riêng** cho YOLOv5. Đặt
`signa-v1.1.zip` ở gốc My Drive và `best.pt` trong
`Signa/experiments/<SOURCE_RUN>/weights/`. Dùng checkpoint do nhóm tạo.

Chạy **Run all** để xuất 640/480 px × FP16/INT8. Hiệu chỉnh INT8 bằng
ảnh train và chấm COCO trên val ở ngưỡng dự đoán 0,001. Kết quả lưu ở
`Signa/exports/<SOURCE_RUN>/`. Khi chạy lại, cấu hình đã lưu đủ tệp,
khớp checksum và cùng thiết lập sẽ được bỏ qua.

In [ ]:
# 1 · Chọn checkpoint nguồn và mount Drive.
from pathlib import Path

from google.colab import drive

SOURCE_RUN = "yolov5n-640-v1.1-main-s0"
REPO_COMMIT = "9659c32487c378867d8ae12cd7e2b79328056c21"
DRIVE = Path("/content/drive/MyDrive")
REPO = Path("/content/signa-tflite-source")
DATASET = Path("/content/v1.1")
ENVIRONMENT = Path("/content/signa-tflite-yolov5")

drive.mount("/content/drive")

In [ ]:
# 2 · Chuẩn bị môi trường xuất CPU, tách khỏi Python của Colab.
import os
import subprocess
import sys


def sh(*arguments, **kwargs):
    subprocess.run([str(arg) for arg in arguments], check=True, **kwargs)


def checkout(url, destination, commit):
    if not (destination / ".git").exists():
        sh("git", "clone", "-q", url, destination)
    sh("git", "-C", destination, "fetch", "-q", "origin")
    sh("git", "-C", destination, "checkout", "-q", "--detach", commit)


checkout("https://github.com/hana200804/DA.git", REPO, REPO_COMMIT)

YOLOV5_COMMIT = "1e7cf74cb64d719b34f7315aa13eab9e32cde672"
YOLOV5_REPO = Path("/content/signa-tflite-yolov5-source")
checkout("https://github.com/ultralytics/yolov5.git", YOLOV5_REPO, YOLOV5_COMMIT)

sh(sys.executable, "-m", "pip", "install", "-q", "uv==0.9.7")
uv = (sys.executable, "-m", "uv")
python = ENVIRONMENT / "bin/python"
if not python.exists():
    sh(*uv, "venv", "--python", "3.11", ENVIRONMENT)
packages = [
    "torch==2.8.0+cpu",
    "torchvision==0.23.0+cpu",
    "ultralytics==8.4.201",
    "ultralytics-thop==2.2.4",
    "numpy==1.26.4",
    "opencv-python==4.11.0.86",
    "pillow==11.3.0",
    "pyyaml==6.0.2",
    "pycocotools==2.0.10",
    "tensorflow==2.15.1",
    "keras==2.15.0",
    "protobuf==4.25.8",
]
requirements = ["-r", YOLOV5_REPO / "requirements.txt"]

sh(
    *uv, "pip", "install", "--python", python,
    "--index", "https://download.pytorch.org/whl/cpu",
    "--index-strategy", "unsafe-best-match",
    *requirements, *packages,
)
sh(*uv, "pip", "check", "--python", python)

In [ ]:
# 3 · Xuất bốn cấu hình, chấm COCO val và ghi kết quả lên Drive.
from time import gmtime, strftime

if SOURCE_RUN in ("", "..") or Path(SOURCE_RUN).name != SOURCE_RUN or "\\" in SOURCE_RUN:
    raise ValueError("SOURCE_RUN phải là tên một thư mục")
output = DRIVE / "Signa/exports" / SOURCE_RUN
output.mkdir(parents=True, exist_ok=True)
command = [
    str(python), "-u", str(REPO / "ai/scripts/export_tflite.py"),
    "--model", "yolov5", "--run", SOURCE_RUN,
    "--drive", str(DRIVE), "--dataset", str(DATASET),
    "--yolov5-repo", str(YOLOV5_REPO),
]
log_path = output / "export.log"
time_format = "%Y-%m-%dT%H:%M:%SZ"
print(f"Nhật ký xuất: {log_path}", flush=True)
with log_path.open("a", encoding="utf-8", buffering=1) as log, subprocess.Popen(
    command,
    cwd="/content",
    # Backend inline của kernel Colab cần gói riêng ngoài venv xuất.
    env=dict(
        os.environ, PYTHONPATH=str(REPO / "ai/src"), YOLO_AUTOINSTALL="false", MPLBACKEND="Agg"
    ),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    encoding="utf-8",
    errors="replace",
) as process:
    try:
        started = strftime(time_format, gmtime())
        print(f"\n--- {started} · {REPO_COMMIT} ---", file=log)
        process.stdout.reconfigure(newline="")
        for line in process.stdout:
            print(line, end="", flush=True)
            # Bỏ các lần ghi đè thanh tiến trình khỏi log, giữ dòng kết thúc.
            if not line.endswith("\r"):
                log.write(line)
        process.wait()
    finally:
        if process.poll() is None:
            # Dừng tiến trình nếu ô bị ngắt hoặc không còn đọc được log.
            process.kill()
            process.wait()
        finished = strftime(time_format, gmtime())
        print(f"\n--- {finished} · mã thoát {process.returncode} ---", file=log)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)

print((output / "summary.csv").read_text(encoding="utf-8"))